## NCAA Seed Prediction - Version 3

**Key insight:** Non-tournament teams (no Bid Type) have ground truth Overall Seed = 0.
Only the 91 tournament teams need a 1-68 seed prediction.

**Approach:**
- Tournament teams (Bid Type = AQ or AL): Ensemble of RF + GBR + Ridge with 48 features, clipped to 1-68
- Non-tournament teams (no Bid Type): predict 0

**Output:** `Output/submission_v3.csv`.

In [ ]:
!pip install pandas numpy scikit-learn -q

In [ ]:
DRIVE_PROJECT_FOLDER = "Kaggle NCAA competition Deadline 15th March"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = f"/content/drive/MyDrive/{DRIVE_PROJECT_FOLDER}/final-four-analytics-challenge-26/Data"
except Exception:
    DATA_DIR = "../final-four-analytics-challenge-26/Data"

import pandas as pd
import numpy as np
import os
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [ ]:
def _path(name_2_0, name_default):
    p2 = os.path.join(DATA_DIR, name_2_0)
    p1 = os.path.join(DATA_DIR, name_default)
    return p2 if os.path.exists(p2) else p1

train = pd.read_csv(_path("NCAA_Seed_Training_Set2.0.csv", "NCAA_Seed_Training_Set.csv"))
test = pd.read_csv(_path("NCAA_Seed_Test_Set2.0.csv", "NCAA_Seed_Test_Set.csv"))
sub = pd.read_csv(_path("submission_template2.0.csv", "submission_template.csv"))
print("Train:", train.shape, "| Test:", test.shape)
print("Seeded in train:", train['Overall Seed'].notna().sum())
print("Tournament in test:", test['Bid Type'].notna().sum(), "of", len(test))

In [ ]:
MONTH_TO_NUM = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
                "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}

def parse_wl(val):
    if pd.isna(val) or val == "" or str(val).strip() == "0-0":
        return np.nan, np.nan, np.nan
    s = str(val).strip()
    parts = s.split("-")
    if len(parts) != 2:
        return np.nan, np.nan, np.nan
    def to_num(x):
        x = x.strip()
        if x in MONTH_TO_NUM:
            return MONTH_TO_NUM[x]
        try:
            return int(x)
        except ValueError:
            return np.nan
    w, l = to_num(parts[0]), to_num(parts[1])
    if np.isnan(w) or np.isnan(l):
        return np.nan, np.nan, np.nan
    total = w + l
    return w, l, (w / total if total > 0 else np.nan)

def add_wl(df, col):
    if col not in df.columns:
        return df
    results = [parse_wl(x) for x in df[col]]
    df = df.copy()
    df[f"{col}_w"] = [r[0] for r in results]
    df[f"{col}_l"] = [r[1] for r in results]
    df[f"{col}_pct"] = [r[2] for r in results]
    return df

for col in ["WL", "Conf.Record", "Non-ConferenceRecord", "RoadWL",
            "Quadrant1", "Quadrant2", "Quadrant3", "Quadrant4"]:
    train = add_wl(train, col)
    test = add_wl(test, col)
print("W-L features parsed.")

In [ ]:
for df in [train, test]:
    df['NET_change'] = df['NET Rank'] - df['PrevNET']
    df['WinPct'] = df['WL_w'] / (df['WL_w'] + df['WL_l'])
    df['Q1_margin'] = df['Quadrant1_w'] - df['Quadrant1_l']
    df['Q12_wins'] = df['Quadrant1_w'].fillna(0) + df['Quadrant2_w'].fillna(0)
    df['Q12_losses'] = df['Quadrant1_l'].fillna(0) + df['Quadrant2_l'].fillna(0)
    df['Q34_wins'] = df['Quadrant3_w'].fillna(0) + df['Quadrant4_w'].fillna(0)
    df['Q34_losses'] = df['Quadrant3_l'].fillna(0) + df['Quadrant4_l'].fillna(0)
    df['is_AQ'] = (df['Bid Type'] == 'AQ').astype(int)
    df['is_AL'] = (df['Bid Type'] == 'AL').astype(int)
    df['has_bid'] = df['Bid Type'].notna().astype(int)
    df['SOS_diff'] = df['NETSOS'] - df['NETNonConfSOS']
    df['NET_x_WinPct'] = df['NET Rank'] * df['WinPct']

all_conf = pd.concat([train['Conference'], test['Conference']]).astype(str).fillna("__NA__")
le = LabelEncoder()
le.fit(all_conf.unique())
train['Conf_enc'] = le.transform(train['Conference'].astype(str).fillna("__NA__"))
test['Conf_enc'] = le.transform(test['Conference'].astype(str).fillna("__NA__"))

features = [
    "NET Rank", "PrevNET", "AvgOppNETRank", "AvgOppNET", "NETSOS", "NETNonConfSOS",
    "WL_w", "WL_l", "WL_pct", "Conf.Record_w", "Conf.Record_l", "Conf.Record_pct",
    "Non-ConferenceRecord_w", "Non-ConferenceRecord_l", "Non-ConferenceRecord_pct",
    "RoadWL_w", "RoadWL_l", "RoadWL_pct",
    "Quadrant1_w", "Quadrant1_l", "Quadrant1_pct",
    "Quadrant2_w", "Quadrant2_l", "Quadrant2_pct",
    "Quadrant3_w", "Quadrant3_l", "Quadrant3_pct",
    "Quadrant4_w", "Quadrant4_l", "Quadrant4_pct",
    "NET_change", "WinPct", "Q1_margin", "Q12_wins", "Q12_losses",
    "Q34_wins", "Q34_losses",
    "is_AQ", "is_AL", "has_bid", "Conf_enc", "SOS_diff", "NET_x_WinPct",
]
features = [c for c in features if c in train.columns and c in test.columns]
print(f"Total features: {len(features)}")

In [ ]:
train_seeded = train[train['Overall Seed'].notna()].copy()
train_seeded['Overall Seed'] = train_seeded['Overall Seed'].astype(int)

X_train = train_seeded[features].values
y_train = train_seeded['Overall Seed'].values
X_test_all = test[features].values

imp = SimpleImputer(strategy='median')
X_train_imp = imp.fit_transform(X_train)
X_test_imp = imp.transform(X_test_all)

print(f"Training on {len(X_train_imp)} seeded teams, predicting {len(X_test_imp)} test teams")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf = RandomForestRegressor(n_estimators=500, max_depth=12, min_samples_leaf=3,
                           random_state=RANDOM_STATE, n_jobs=-1)
gbr = GradientBoostingRegressor(n_estimators=400, max_depth=5, learning_rate=0.05,
                                min_samples_leaf=5, random_state=RANDOM_STATE)
ridge = Ridge(alpha=1.0)

print("Cross-validation RMSE on seeded teams:")
for name, model in [("RF", rf), ("GBR", gbr), ("Ridge", ridge)]:
    cv_pred = cross_val_predict(model, X_train_imp, y_train, cv=kf)
    rmse = np.sqrt(np.mean((y_train - cv_pred)**2))
    print(f"  {name}: {rmse:.4f}")

rf.fit(X_train_imp, y_train)
gbr.fit(X_train_imp, y_train)
ridge.fit(X_train_imp, y_train)

pred_ensemble = (0.4 * rf.predict(X_test_imp) +
                 0.4 * gbr.predict(X_test_imp) +
                 0.2 * ridge.predict(X_test_imp))
print("Ensemble fitted.")

In [ ]:
tourney_mask = test['Bid Type'].notna()

predictions = np.zeros(len(test), dtype=int)
predictions[tourney_mask.values] = np.clip(
    np.round(pred_ensemble[tourney_mask.values]), 1, 68
).astype(int)

out = sub[['RecordID']].copy()
out['Overall Seed'] = predictions

print(f"Tournament teams ({tourney_mask.sum()}): range {predictions[tourney_mask.values].min()}-{predictions[tourney_mask.values].max()}")
print(f"Non-tournament ({(~tourney_mask).sum()}): all = {predictions[~tourney_mask.values][0]}")

est_rmse = np.sqrt(91 * 6.0**2 / 451)
print(f"Estimated RMSE: ~{est_rmse:.1f}")

In [ ]:
try:
    out_dir = os.path.join(os.path.dirname(DATA_DIR), '..', 'Output')
except Exception:
    out_dir = '../Output'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'submission_v3.csv')
out.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Rows: {len(out)}")
print()
print(out.head(20))

try:
    from google.colab import files
    files.download(out_path)
    print("Download started.")
except Exception:
    pass